In [ ]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import pickle
import numpy as np
import matplotlib.pyplot as plt
import math
import json
from datetime import date
plt.rcParams["figure.figsize"] = (24,18)

from ultralytics import YOLO
from cv_utils import *

In [ ]:
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')
model = YOLO(os.path.join(model_path, "yolov8n.pt"))

In [ ]:
data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')
vid_name =  'synth_train'

In [ ]:
images_path = os.path.join(data_path, 'images' + '/' + vid_name)
print(os.path.exists(images_path))
images_path
image_path = os.path.join(images_path, os.listdir(images_path)[1])

rgb_image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)

plt.figure()
plt.imshow(rgb_image[:,:,::-1])

In [ ]:
results = model.predict(rgb_image)

for result in results:
    boxes = result.boxes
    for box in boxes:
        coord = box.xyxy.cpu().detach().int().tolist()[0]
        class_id = box.cls[0].item()
        print(class_id)
        print("Object type:", box.cls)
        
        print(coord)
        (startX, startY, endX, endY) = coord
        cv2.rectangle(rgb_image, (startX, startY), (endX, endY),
    				(255, 0, 0), 2)
        
        
        
plt.figure()
plt.imshow(rgb_image[:,:,::-1])


In [ ]:
dest_path = os.path.join(data_path, 'annotated_frames' + '/' + vid_name)
if not os.path.exists(dest_path):
    os.mkdir(dest_path)


visualize_path = os.path.join(dest_path, "annotated_frames")
if not os.path.exists(visualize_path):
    os.mkdir(visualize_path)

for image_name in os.listdir(images_path):
    frame = os.path.join(images_path, image_name)

    # Run YOLOv8 inference on the frame
    results = model(frame, classes=[0, 32])

    # Visualize the results on the frame
    annotated_frame = results[0].plot()

    # Display the annotated frame
    cv2.imwrite(os.path.join(visualize_path, image_name), annotated_frame)

ref_image = cv2.imread(os.path.join(visualize_path, os.listdir(visualize_path)[0]), cv2.IMREAD_UNCHANGED)
h, w, _ = ref_image.shape
video= cv2.VideoWriter(os.path.join(visualize_path, vid_name+'.mp4'), cv2.VideoWriter_fourcc(*'mp4v'), 3, (w,h))
for image_name in os.listdir(visualize_path):
    frame = cv2.imread(os.path.join(visualize_path, image_name), cv2.IMREAD_UNCHANGED)
    video.write(frame)
video.release()
